# Week 67, Day 5: Choosing a concurrency model

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
tasks = 100
cpu_ms, io_ms = 5, 95             # each task: a little computing, a lot of waiting
cores = 4

sequential = tasks * (cpu_ms + io_ms)
threads_or_async = tasks * cpu_ms + io_ms          # waits overlap; computing still one at a time
processes = (tasks * cpu_ms) / cores + io_ms * tasks / cores
print(sequential, threads_or_async, processes)

In [ ]:
def choose(slow_enough, io_bound, many_connections=False, numeric=False, free_threaded=False):
    if not slow_enough:
        return "sequential"
    if io_bound:
        return "asyncio" if many_connections else "threads"
    if numeric:
        return "NumPy"
    return "threads (free-threaded build)" if free_threaded else "processes"

cases = {
    "a script that runs in 2 s once a day": dict(slow_enough=False, io_bound=True),
    "download 200 files": dict(slow_enough=True, io_bound=True),
    "a chat server with 10,000 users": dict(slow_enough=True, io_bound=True, many_connections=True),
    "resize 5,000 images in pure Python": dict(slow_enough=True, io_bound=False),
    "statistics on a million readings": dict(slow_enough=True, io_bound=False, numeric=True),
}
for name, args in cases.items():
    print(f"{name:40} -> {choose(**args)}")

In [ ]:
def estimate(n, cpu, io, cores, model):
    if model == "sequential":
        return n * (cpu + io)
    if model in ("threads", "asyncio"):
        return n * cpu + io
    if model == "processes":
        per_process = -(-n // cores)
        return per_process * (cpu + io)
    raise ValueError(model)

for cpu, io in [(5, 95), (95, 5)]:
    times = {m: estimate(100, cpu, io, 4, m) for m in ("sequential", "threads", "processes")}
    print(f"cpu={cpu} io={io}:", times, "best:", min(times, key=times.get))

In [ ]:
def hybrid_time(n, cpu, io, cores):
    "Event loop overlaps every wait; a process pool runs the computing across cores."
    return -(-n // cores) * cpu + io

def threads_time(n, cpu, io):
    return n * cpu + io

for cpu, io in [(5, 95), (50, 50), (95, 5)]:
    print(f"cpu={cpu} io={io}: threads {threads_time(100, cpu, io)}, asyncio + process pool {hybrid_time(100, cpu, io, 4)}")

In [ ]:
def throughput(stages):
    "stages: name -> (workers, time per item)."
    rates = {name: workers / t for name, (workers, t) in stages.items()}
    bottleneck = min(rates, key=rates.get)
    return rates, bottleneck, rates[bottleneck]

stages = {"download": (8, 0.4), "parse": (2, 0.05), "store": (4, 0.1)}
rates, slowest, total = throughput(stages)
print({k: round(v, 1) for k, v in rates.items()}, "bottleneck:", slowest, round(total, 1), "items per second")
stages["download"] = (32, 0.4)
print(throughput(stages)[1:])

In [ ]:
measured = {1: 100.0, 2: 52.0, 4: 28.0, 8: 18.0, 16: 15.0}

for n, t in measured.items():
    speedup = measured[1] / t
    print(f"{n:2} workers: speed-up {speedup:4.1f}, efficiency {speedup / n:.2f}")

worth_it = max(n for n, t in measured.items() if measured[1] / t / n >= 0.6)
print("the most workers still at least 60% efficient:", worth_it)

> **Key idea.** Stay sequential unless it's slow enough to matter; profile to see whether time goes on waiting or computing. Waiting: threads for modest numbers of tasks or blocking libraries, asyncio for thousands of connections with async libraries. Computing: NumPy for array work, processes for pure Python (free-threaded threads once libraries support them). Threads are cheap but race; processes are isolated but costly and copy data; asyncio tasks are tiny but need async code throughout. Combine models with asyncio.to_thread, run_in_executor with a process pool, queues between owners of state, and several processes each with an event loop. A pipeline runs at the throughput of its slowest stage, so balance workers per stage and bound queues. Measure speed-up T(1)/T(n) and efficiency S(n)/n, stop adding workers when efficiency falls too low, and skip concurrency when it doesn't buy what you need.

### Exercise: A decision function

Write `choose(work)` for a dict with keys `seconds` (how long it takes now), `target` (how long it may take), `io_fraction` (0 to 1: the share of time spent waiting), `connections` (how many simultaneous tasks), `numeric` (bool) and `async_libraries` (bool: async versions of the libraries it needs exist). Rules, in order: `"sequential"` if `seconds <= target`; for `io_fraction >= 0.5`, `"asyncio"` if there are at least 1000 connections **and** async libraries exist, otherwise `"threads"`; for CPU-bound work, `"numpy"` if numeric, otherwise `"processes"`. Then `reasons(work)`: a list of short strings explaining each decision taken on the way, from these: `"fast enough"`, `"mostly waiting"`, `"mostly computing"`, `"many connections"`, `"no async libraries"`, `"array work"`.

In [ ]:
def choose(work):
    ___

def reasons(work):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_choose():\n    \"choosing a model\"\n    base = {\"seconds\": 60, \"target\": 10, \"io_fraction\": 0.9, \"connections\": 50, \"numeric\": False, \"async_libraries\": True}\n    assert choose(dict(base, seconds=5)) == \"sequential\" and reasons(dict(base, seconds=5)) == [\"fast enough\"]\n    assert choose(base) == \"threads\" and reasons(base) == [\"mostly waiting\"]\n    many = dict(base, connections=5000)\n    assert choose(many) == \"asyncio\" and reasons(many) == [\"mostly waiting\", \"many connections\"]\n    blocked = dict(many, async_libraries=False)\n    assert choose(blocked) == \"threads\" and reasons(blocked) == [\"mostly waiting\", \"many connections\", \"no async libraries\"]\n    cpu = dict(base, io_fraction=0.1)\n    assert choose(cpu) == \"processes\" and choose(dict(cpu, numeric=True)) == \"numpy\"\n    assert reasons(dict(cpu, numeric=True)) == [\"mostly computing\", \"array work\"]")

### Exercise: Estimating each model

For `n` independent tasks, each computing for `cpu` and waiting for `io`, write `estimate(n, cpu, io, model, cores=1, workers=None)`:

- `"sequential"`: `n * (cpu + io)`;
- `"threads"` with `workers` threads (default `n`): computing is one at a time (the GIL); tasks are started in batches of `workers`, each batch taking `batch_size * cpu + io`; total is the sum over batches;
- `"asyncio"`: like threads with `workers = n` (every wait overlaps);
- `"processes"` with `cores` processes, each working through its share of tasks one after another: the largest share times `(cpu + io)`.

Then `best(n, cpu, io, cores)`: the model with the smallest estimate among those four (threads with the default number of workers), ties in the order listed.

In [ ]:
def estimate(n, cpu, io, model, cores=1, workers=None):
    ___

def best(n, cpu, io, cores):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_estimate():\n    \"cost models\"\n    assert estimate(100, 5, 95, \"sequential\") == 10000\n    assert estimate(100, 5, 95, \"asyncio\") == 595 and estimate(100, 5, 95, \"threads\") == 595\n    assert estimate(100, 5, 95, \"threads\", workers=10) == 10 * (50 + 95)\n    assert estimate(100, 5, 95, \"processes\", cores=4) == 2500 and estimate(10, 1, 1, \"processes\", cores=3) == 8\n    assert best(100, 5, 95, 4) == \"threads\" and best(100, 95, 5, 4) == \"processes\" and best(1, 1, 1, 1) == \"sequential\"")

### Exercise: How many connections fit?

Using the per-unit memory costs in `COSTS_KB` (a rough guide: a thread's stack, a process's interpreter, an asyncio task), write `memory_mb(connections, model)`: the memory in MB (KB divided by 1024, rounded to 1 place) for one unit per connection, plus a fixed `BASE_KB` for the program itself; and `max_connections(budget_mb, model)`: the most connections that fit in the budget (0 if not even the base fits).

In [ ]:
COSTS_KB = {"thread": 256, "process": 30_000, "task": 2}
BASE_KB = 20_000

def memory_mb(connections, model):
    ___

def max_connections(budget_mb, model):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memory():\n    \"memory per connection\"\n    assert memory_mb(0, \"task\") == 19.5 and memory_mb(10_000, \"task\") == 39.1\n    assert memory_mb(1000, \"thread\") == 269.5 and memory_mb(10, \"process\") == 312.5\n    assert max_connections(100, \"task\") == 41_200 and max_connections(100, \"thread\") == 321\n    assert max_connections(100, \"process\") == 2 and max_connections(10, \"task\") == 0")

### Exercise: Finding the bottleneck

A pipeline's stages are given as a list of `(name, workers, seconds_per_item)`. Write `rates(stages)`: a dict from stage name to items per second (`workers / seconds_per_item`, rounded to 2 places); `bottleneck(stages)`: the name of the slowest stage (the first on ties); and `balance(stages, budget)`: given a total number of workers to share out, return a dict from stage name to its number of workers that **maximises** the pipeline's throughput: start with one worker per stage, then give each remaining worker, one at a time, to the current bottleneck (each stage keeps its own `seconds_per_item`).

In [ ]:
def rates(stages):
    ___

def bottleneck(stages):
    ___

def balance(stages, budget):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipeline():\n    \"pipelines\"\n    stages = [(\"download\", 8, 0.4), (\"parse\", 2, 0.05), (\"store\", 4, 0.1)]\n    assert rates(stages) == {\"download\": 20.0, \"parse\": 40.0, \"store\": 40.0} and bottleneck(stages) == \"download\"\n    assert bottleneck([(\"a\", 1, 1), (\"b\", 1, 1)]) == \"a\"\n    plan = balance(stages, 10)\n    assert sum(plan.values()) == 10 and plan == {\"download\": 7, \"parse\": 1, \"store\": 2}\n    assert balance([(\"x\", 0, 2), (\"y\", 0, 1)], 3) == {\"x\": 2, \"y\": 1}")

### Exercise: Speed-up and efficiency

Given measured times as a dict from number of workers to seconds (it always includes 1), write `speedups(times)`: a dict from workers to speed-up `T(1) / T(n)` rounded to 2 places; `efficiencies(times)`: a dict from workers to `speedup / n` rounded to 2 places; and `worth_it(times, threshold)`: the largest worker count whose efficiency is at least `threshold`.

In [ ]:
def speedups(times):
    ___

def efficiencies(times):
    ___

def worth_it(times, threshold):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_efficiency():\n    \"measuring scaling\"\n    t = {1: 100.0, 2: 52.0, 4: 28.0, 8: 18.0, 16: 15.0}\n    assert speedups(t) == {1: 1.0, 2: 1.92, 4: 3.57, 8: 5.56, 16: 6.67}\n    assert efficiencies(t) == {1: 1.0, 2: 0.96, 4: 0.89, 8: 0.69, 16: 0.42}\n    assert worth_it(t, 0.6) == 8 and worth_it(t, 0.9) == 2 and worth_it(t, 1.0) == 1")

### Bug hunt: Threads for the wrong job

The rule picks threads whenever the work is slow, so this CPU-bound job gets threads, which the GIL stops from computing in parallel, and this prints `threads`. Choose processes for CPU-bound work and threads only for I/O-bound work, so it prints `processes`.

In [ ]:
def choose(slow, io_bound):
    if not slow:
        return "sequential"
    return "threads"

print(choose(slow=True, io_bound=False))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"processes for computing\"\n    expect_output(\"processes\")")

### Bug hunt: Efficiency upside down

Efficiency is speed-up divided by the number of workers, but this divides the **time** by the workers, giving a meaningless number: this prints `7.0`. Compute the speed-up first, `T(1) / T(n)`, then divide it by `n`, so it prints `0.89`.

In [ ]:
times = {1: 100.0, 4: 28.0}
n = 4
print(round(times[n] / n, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"speed-up over workers\"\n    expect_output(\"0.89\")")

### Bug hunt: The fastest stage isn't the limit

The pipeline's throughput is limited by its **slowest** stage, but this takes the fastest, so it reports `40.0` items per second. Take the minimum stage rate so it prints `20.0`.

In [ ]:
stages = {"download": 8 / 0.4, "parse": 2 / 0.05, "store": 4 / 0.1}
print(max(stages.values()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the bottleneck\"\n    expect_output(\"20.0\")")

### Puzzle: Computing without blocking the loop

Arrange the lines of a coroutine that awaits I/O on the event loop but runs a CPU-heavy function in a process pool (real `asyncio`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the model for a chat server with 20,000 connections and async libraries
q = ___   # the model for pure-Python image processing on 8 cores (normal build)
r = ___   # throughput of a pipeline with stage rates 10, 4 and 7 items per second
s = ___   # efficiency with 4 workers when T(1) = 80 and T(4) = 25
t = ___   # True or False: asyncio.run can be called inside a coroutine already running on a loop

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"asyncio\" and q == \"processes\" and r == 4, \"p, q and r\"\n    assert s == 0.8 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a server of event loops

Simulate a server built the usual way: `processes` worker processes, each running an event loop. Requests are `(arrival_time, cpu, io)`: each needs `cpu` units of computing (holding its process's single core, since the GIL serialises Python code in a process) and then `io` units of waiting (which overlaps freely with anything). Write `serve(requests, processes, limit)`:

- requests are dispatched in arrival order (ties: list order), each to the process with the **fewest unfinished requests** (lowest-numbered on ties) at its arrival time;
- a process accepts at most `limit` unfinished requests; a request arriving when every process is full waits in a global queue and is dispatched, in order, the moment some request finishes;
- within a process, computing is done one request at a time, in the order they reached that process; a request's waiting starts as soon as its computing ends.

Return a list of finish times in input order, and write `slowest_wait(requests, processes, limit)`: the largest `finish - arrival - cpu - io` (time spent queuing or waiting for the core).

In [ ]:
import heapq

def serve(requests, processes, limit):
    ___

def slowest_wait(requests, processes, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_server():\n    \"processes of event loops\"\n    assert serve([(0, 2, 10), (0, 2, 10)], 1, 10) == [12, 14]\n    assert serve([(0, 2, 10), (0, 2, 10)], 2, 10) == [12, 12]\n    assert serve([(0, 1, 5), (0, 1, 5), (0, 1, 5)], 1, 2) == [6, 7, 12]\n    reqs = [(0, 3, 1), (1, 3, 1), (2, 3, 1), (2, 1, 20)]\n    assert serve(reqs, 2, 5) == [4, 5, 7, 25]\n    assert slowest_wait(reqs, 2, 5) == 2 and slowest_wait([(0, 1, 1)], 1, 1) == 0\n    assert serve([], 3, 2) == []")

## Warm-up

For I/O-heavy work, overlapping the waits (threads or `asyncio`) wins easily: 595 ms instead of 10 seconds. Four processes that each handle tasks one at a time help much less here (2,500 ms). Change the numbers so the work is mostly computing, and the ranking flips. The shape of the work decides.

## The questions that decide

Ask, in order:

1. **Is it slow enough to matter?** If not, stay sequential: it's simpler, easier to test, and has no races (Week 65's first rule).
2. **Where does the time go: waiting or computing?** Profile it (Week 65).
3. **Waiting (I/O-bound)**:
    - **A modest number of tasks** (up to a few hundred), or blocking libraries you can't replace: **threads** (`ThreadPoolExecutor`).
    - **Many simultaneous connections** (thousands), or a framework that's already async: **`asyncio`**. It needs async libraries throughout.
4. **Computing (CPU-bound)**:
    - **Numeric work on arrays**: **NumPy** first (Week 65), which may make concurrency unnecessary.
    - **Pure-Python computation**: **processes** (`ProcessPoolExecutor`), with sizeable chunks of work and small data to send; or **threads on a free-threaded build** (Python 3.13+), once your libraries support it.
5. **Both**: combine (next sections): an event loop or thread pool for the waiting, a process pool for the computing.

Two more constraints often decide in practice: **shared state** (threads share memory easily; processes need copying or explicit sharing; `asyncio` shares freely within one thread) and the **ecosystem** (a library you depend on may only be blocking, or only async).

## What each model costs

Rough orders of magnitude (they vary a lot by machine and version, so measure your own):

| | start-up | memory per unit | switching | good scale |
|---|---|---|---|---|
| thread | tens of microseconds | an OS stack (reserved space; real use is often tens of KB) | OS decides, any time | up to hundreds |
| process | milliseconds (spawn: tens of ms) | a whole interpreter: many MB | OS decides | up to the number of cores |
| asyncio task | a few microseconds | about a KB or two | only at await | tens of thousands |

And the costs that aren't in the table: threads bring **races** and the need for locks; processes bring **pickling** and copied data; `asyncio` brings **function colouring** (async code can only be awaited from async code, so it spreads through a program) and the need for async libraries everywhere.

A simple time model for `n` tasks, each with `cpu` time computing and `io` time waiting, on `c` cores:

- sequential: `n * (cpu + io)`;
- threads or asyncio (with the GIL): `n * cpu + io` when the waits all overlap (computing is still one at a time);
- processes: about `n * (cpu + io) / c` when each process works through its share sequentially.

## Combining models

Real programs often need more than one model. The safe combinations:

- **`asyncio` + a thread pool**: `await asyncio.to_thread(blocking_call, ...)` for a library that isn't async (a database driver, file system work, an image library). The event loop stays responsive.
- **`asyncio` + a process pool**: `await loop.run_in_executor(process_pool, cpu_heavy, data)` for computation, so the event loop handles I/O while other cores compute.
- **Threads + `queue.Queue`**: a thread pool doing blocking I/O, passing results to one thread that owns shared state. No locks needed for that state.
- **Several processes, each with an event loop (or thread pool)**: how production web servers scale across cores (a "pre-fork" or "worker" model: Gunicorn, Uvicorn workers). A load balancer or the operating system spreads connections across them.

Rules that keep combinations sane: decide which thread or process **owns** each piece of state; pass messages between models (queues, futures) rather than sharing objects; never call blocking code directly in a coroutine; and never call `asyncio.run` inside an already running loop (use `await` instead, or `asyncio.run_coroutine_threadsafe` to submit work to a loop from another thread).

```
import asyncio
from concurrent.futures import ProcessPoolExecutor

def analyse(data):                       # CPU-heavy, pure Python
    return sum(x * x for x in data)

async def handle(request, pool):
    data = await fetch(request)          # I/O: awaited on the event loop
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(pool, analyse, data)   # computing in another process
    await save(result)                   # I/O again

async def main(requests):
    with ProcessPoolExecutor() as pool:
        await asyncio.gather(*(handle(r, pool) for r in requests))
```

## Pipelines and bottlenecks

Many concurrent programs are **pipelines**: download, then parse, then store. Each **stage** has its own work per item and its own number of workers, and the whole pipeline can go no faster than its **slowest stage**, the **bottleneck**:

- a stage's **throughput** is `workers / time_per_item` items per unit of time;
- the pipeline's throughput is the **minimum** over its stages;
- adding workers to any stage **except** the bottleneck changes nothing; items just pile up before the bottleneck (so bound the queues between stages, Week 66's back-pressure).

This also tells you which model each stage needs: I/O stages get threads or tasks, CPU stages get processes, and the number of workers per stage is chosen to balance throughputs.

## Measuring, and knowing when to stop

After choosing, **measure** (Week 65): time the whole job with 1, 2, 4, 8 ... workers and compute:

- **speed-up** `S(n) = T(1) / T(n)`;
- **efficiency** `E(n) = S(n) / n`: the fraction of each added worker that's actually used.

Efficiency usually falls as `n` grows (Amdahl's serial part, overheads, contention for locks, memory bandwidth, the network). Stop adding workers where efficiency drops below what's worth paying for, often around 0.5 to 0.7.

And know when **not** to use concurrency at all: when the job is fast enough already, when the work is mostly serial, when correctness is hard to guarantee and the gain is small, or when a better algorithm or NumPy would give a bigger speed-up for less complexity. Concurrency multiplies the difficulty of testing and debugging; spend that cost only where it buys something you need.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a program you'd like to build that does several things at once. Write down where its time goes, which model fits each part, and what the bottleneck would be. Tomorrow is the Forge: an async job system, built on your own event loop, using this week's patterns together.